# 2 — Rebuild raw transactions from `problem_info.csv`

`problem_info.csv` keeps each attempt's full transaction sequence, so a slice of raw
rows can be rebuilt without the multi-GB export -- useful for sharing an inspectable
example.

Every rebuilt attempt is checked back against the columns `problem_info.csv` derived
from the original rows. `tokenize_steps` and `time_info` are the *same* functions the
build uses, which is what makes the check a genuine round trip rather than a
comparison between two copies of the same logic.

## Imports

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Sequence, Tuple
import argparse
import csv
import io
import os
import sys

import numpy as np
import pandas as pd

## Paths

In [ ]:
# Paths. Everything lives inside this folder, so the notebook is self-contained.
NOTEBOOK_DIR = Path.cwd()

DATASET_DIR = Path(os.environ.get("PC_DATASET_DIR", NOTEBOOK_DIR / "dataset"))
OUTPUT_DIR = Path(os.environ.get("PC_OUTPUT_DIR", NOTEBOOK_DIR / "outputs"))
RESULTS_DIR = Path(os.environ.get("PC_RESULTS_DIR", NOTEBOOK_DIR / "results"))


def workspace_dir(workspace, create=False):
    """Output directory holding one workspace's data.pkl and problem_info.csv."""
    path = OUTPUT_DIR / workspace
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


print(f"dataset: {DATASET_DIR}\noutputs: {OUTPUT_DIR}\nresults: {RESULTS_DIR}")

## Workspace configuration

Shared constants and the per-workspace configuration.

`ratio_proportion_change3` and `ratio_proportion_change4` run the same pipeline; this
module holds everything that differs between them, so the stage code stays generic.

In [ ]:
# ── Structural steps ──────────────────────────────────────────────────────────────────
# Steps MATHia renders without a KC of its own. Both workspaces share these 11; change4
# adds three more.
STRUCTURAL_STEPS = [
    "OptionalTask_1", "EquationAnswer", "NumeratorFactor", "DenominatorFactor",
    "OptionalTask_2", "FirstRow1:1", "FirstRow1:2", "FirstRow2:1", "FirstRow2:2",
    "SecondRow", "ThirdRow",
]
CHANGE4_EXTRA_STRUCTURAL_STEPS = [
    "PercentChange", "NumeratorLabel1", "DenominatorLabel1",
]

# The two optional-task paths. OPT_STEP*[0] is the entry step, the rest are its substeps.
OPT_STEP1 = ["OptionalTask_1", "EquationAnswer", "NumeratorFactor", "DenominatorFactor"]
OPT_STEP2 = ["OptionalTask_2", "FirstRow1:1", "FirstRow1:2", "FirstRow2:1", "FirstRow2:2",
             "SecondRow", "ThirdRow"]
OPT_ALL_STEPS = set(OPT_STEP1 + OPT_STEP2)
OPT_SUBSTEPS = set(OPT_STEP1[1:] + OPT_STEP2[1:])

ER_PATH_STEPS = frozenset(OPT_STEP1)              # equivalent-ratio (fraction factor)
ME_PATH_STEPS = frozenset(OPT_STEP2)              # means-and-extremes

# ── KCs borrowed for KC-less structural steps ─────────────────────────────────────────
# Labels are taken verbatim from the prop1/prop2 workspaces so a skill keeps one name
# across all four workspaces.
STRUCTURAL_KC_CHANGE3 = {
    "FirstRow1:1": "enter first extreme in equation-1",
    "FirstRow1:2": "enter second extreme in equation-1",
    "FirstRow2:1": "enter first mean in equation-1",
    "FirstRow2:2": "enter second mean in equation-1",
    "SecondRow":   "calculate product of means or extremes-1",
}
STRUCTURAL_KC_CHANGE4 = {
    **STRUCTURAL_KC_CHANGE3,
    "PercentChange":        "identify percent change as increase or decrease-1",
    "NumeratorLabel1":      "enter proportion label in numerator-1",
    "DenominatorLabel1":    "enter proportion label in denominator-1",
    "DenominatorQuantity1": "enter given original amount in proportion-1",
}

# ── Conditional structural KCs ────────────────────────────────────────────────────────
# These four steps get their KC per problem, from the problem's numbers (see the metadata stage).
NUMFACTOR_INT_KC   = "enter numerator of form of 1-1 for integer factor"
NUMFACTOR_FRAC_KC  = "enter numerator of form of 1-1 for fractional factor"
DENFACTOR_INT_KC   = "enter denominator of form of 1-1 for integer factor"
DENFACTOR_FRAC_KC  = "enter denominator of form of 1-1 for fractional factor"
EQANSWER_PART_KC   = "calculate part in proportion with fractions-1"
EQANSWER_TOTAL_KC  = "calculate total in proportion with fractions-1"
THIRDROW_SIMPLE_KC = "calculate solution with means and extremes using simple numbers-1"
THIRDROW_DIFF_KC   = "calculate solution with means and extremes using difficult numbers-1"

STRUCTURAL_KC_CONDITIONAL_STEPS = {
    "NumeratorFactor", "DenominatorFactor", "EquationAnswer", "ThirdRow",
}

# ── Synthetic KCs appended to the Q-matrix ────────────────────────────────────────────
# Recognize-*: did the student take the path the problem's numbers call for?
RECOGNIZE_ER_KC = RECOGNIZE_ER_STEP = "Recognize-ER"
RECOGNIZE_ME_KC = RECOGNIZE_ME_STEP = "Recognize-ME"
# SelectOptimalStrategy: the same judgement as one observation per problem, appended at the
# end; the BeforeFA variant places it just before the final answer instead.
SOS_KC = SOS_STEP = "SelectOptimalStrategy"
SOS_BFA_KC = SOS_BFA_STEP = "SelectOptimalStrategyBeforeFA"

# ── Outcomes ──────────────────────────────────────────────────────────────────────────
GENUINE_OUTCOMES = {"OK", "ERROR"}
CORRECT_OUTCOME = "OK"

# ── Metadata scenario fields ──────────────────────────────────────────────────────────
# A subtype is the null pattern over these fields: which quantities the problem withholds.
CHANGE_FIELDS = [
    ("ppc-scenario_initial-amount", "initial"),
    ("ppc-scenario_final-amount",   "final"),
    ("ppc-scenario_change-amount",  "change"),
    ("ppc-scenario_percent-change", "pct"),
]
PROP_FIELDS = [
    ("ppc-scenario_percent",      "pct"),
    ("ppc-scenario_total-amount", "total"),
    ("ppc-scenario_part-amount",  "part"),
]

# change3 problem types, read off the metadata null pattern.
PROB_TYPE_CHANGE_AMOUNT     = "change_amount"      # pct given, change withheld
PROB_TYPE_CHANGE_PERCENT_IF = "change_percent_IF"  # initial + final given
PROB_TYPE_CHANGE_PERCENT_IC = "change_percent_IC"  # initial + change given
PROB_TYPE_CHANGE_PERCENT_FC = "change_percent_FC"  # final + change given

# Columns the transaction reconstruction writes back out, in MATHia's order.
MATHIA_COLUMNS = [
    "Anon Student Id", "Time", "Problem Name", "Step Name", "Attempt At Step",
    "Outcome", "Action", "Help Level", "KC Model(MATHia)",
    "CF (Skill Previous p-Known)", "CF (Skill New p-Known)", "CF (Etalon)",
    "CF (Is StepByStep)", "CF (Encounter)", "CF (Is Review Mode)",
    "CF (Is Autofilled)", "CF (Anon Class Id)", "CF (Anon School Id)",
    "CF (Workspace Progress Status)",
]


@dataclass(frozen=True)
class Workspace:
    """One MATHia workspace and the choices its pipeline run makes."""

    name: str
    dataset_file: str                       # raw transaction export, in DATASET_DIR
    metadata_file: str                      # problem text + scenario metadata
    structural_steps: List[str]             # KC-less steps, excluded from mastery skills
    structural_kc: Dict[str, str]           # step -> borrowed KC, unconditional steps
    metadata_columns: List[str]             # scenario columns to read
    prob_type_source: str                   # "metadata" (change3) or "steps" (change4)
    proportion_class_subtypes: bool = False  # split subtypes by problem-class (change4)
    coerce_metadata_numeric: bool = False   # some exports store scenario values as text
    # Steps exempt from the "mastery steps must carry a KC" rule, and the step whose
    # presence marks the problems where the exemption applies.
    mastery_kc_exempt: Tuple[Tuple[str, str], ...] = ()
    min_problems: int = 4                   # per-student problem count bounds
    max_problems: int = 40
    columns: List[str] = field(default_factory=lambda: list(MATHIA_COLUMNS))

    @property
    def structural_kc_steps(self) -> set:
        """Structural steps that end up in the Q-matrix, flat map plus conditional."""
        return set(self.structural_kc) | STRUCTURAL_KC_CONDITIONAL_STEPS


CHANGE3 = Workspace(
    name="ratio_proportion_change3",
    dataset_file="MATHia_2223_deidentified_ratio_proportion_change3_large_sample.csv",
    metadata_file="ratio_proportion_change3_all_problems_text_and_metadata.csv",
    structural_steps=STRUCTURAL_STEPS,
    structural_kc=STRUCTURAL_KC_CHANGE3,
    metadata_columns=["lms-id"] + [c for c, _ in CHANGE_FIELDS],
    prob_type_source="metadata",
)

CHANGE4 = Workspace(
    name="ratio_proportion_change4",
    dataset_file="MATHia_2223_deidentified_ratio_proportion_change4_large_sample.csv",
    metadata_file="ratio_proportion_change4_all_problems_text_and_metadata_corrected.csv",
    structural_steps=STRUCTURAL_STEPS + CHANGE4_EXTRA_STRUCTURAL_STEPS,
    structural_kc=STRUCTURAL_KC_CHANGE4,
    metadata_columns=(["lms-id", "problem-class"]
                      + [c for c, _ in CHANGE_FIELDS] + [c for c, _ in PROP_FIELDS]),
    prob_type_source="steps",
    proportion_class_subtypes=True,
    coerce_metadata_numeric=True,
    # DenominatorQuantity1 is KC-less in change4's _percentChange problems.
    mastery_kc_exempt=(("DenominatorQuantity1", "PercentChange"),),
)

WORKSPACES: Dict[str, Workspace] = {ws.name: ws for ws in (CHANGE3, CHANGE4)}

# Accept the short names used on the command line and in the notebooks.
ALIASES = {"change3": CHANGE3.name, "change4": CHANGE4.name}


def get_workspace(name: str) -> Workspace:
    """Look up a workspace by full name or short alias."""
    key = ALIASES.get(name, name)
    if key not in WORKSPACES:
        raise KeyError(f"unknown workspace {name!r}; "
                       f"choose from {sorted(WORKSPACES) + sorted(ALIASES)}")
    return WORKSPACES[key]


def compute_label_opt(step_set: Sequence[str]) -> int:
    """Classify optional-task engagement: 0 = none ... 8 = both paths fully completed."""
    step_set = set(step_set)
    all_opt1 = all(opt in step_set for opt in OPT_STEP1)
    any_opt1 = any(opt in step_set for opt in OPT_STEP1[1:])
    all_opt2 = all(opt in step_set for opt in OPT_STEP2)
    any_opt2 = any(opt in step_set for opt in OPT_STEP2[1:])
    label = 0
    if any_opt1:
        label = 2
    if all_opt1:
        label = 1
    if any_opt2:
        label = 4
    if all_opt2:
        label = 3
    if any_opt1 and any_opt2:
        label = 5
    if any_opt1 and all_opt2:
        label = 6
    if all_opt1 and any_opt2:
        label = 7
    if all_opt1 and all_opt2:
        label = 8
    return label


def recognize_response(er_me: int, has_er: bool, has_me: bool) -> int:
    """1 if the student took only the path the problem calls for, 0 if not, -1 if neither."""
    if not (has_er or has_me):
        return -1
    if er_me == 0:
        return 1 if (has_er and not has_me) else 0
    if er_me == 1:
        return 1 if (has_me and not has_er) else 0
    return -1

## The side table, and the step/time helpers it shares

Stage 5 -- problem_info.csv: one row per (student, problem), in solve order.

data.pkl carries what the models consume; this file carries everything else about the same
problem attempt -- timing, optional-task engagement, the full transaction token sequence.
The token sequence is lossless enough that notebook 2 can rebuild the raw
transactions from it.

In [ ]:
HEADER = [
    "school", "progress", "student", "count", "total_problems",
    "problem", "prob_type", "scenario", "er_me", "etalon_value",
    "label_opt", "all_opt_correct", "fa_correctness_after_opt", "n_tokenized_steps",
    "n_original_steps", "original_steps", "tokenized_steps", "kcs_skills",
    "new_skills", "fa_skill_index", "seq_time_vec", "opt_step_time",
    "non_opt_step_time", "actual_fa_opt_time", "tot_prob_time",
    "recognize_response",
]

# Columns the transaction token sequence is built from.
TRANSACTION_COLS = [
    "Anon Student Id", "Problem Name", "Step Name", "Action", "Attempt At Step",
    "Help Level", "Outcome", "Time", "KC Model(MATHia)",
    "CF (Skill Previous p-Known)", "CF (Skill New p-Known)",
]


def tokenize_steps(steps, actions, outcomes) -> str:
    """Collapse a transaction sequence into one token per step, up to the final answer.

    Non-optional steps become `<step>-<code>` where the code is the worst thing that
    happened on them: 0 = clean, 1 = hint used, 2 = wrong attempt. Optional-task steps keep
    their bare name. Returns "" for problems where no optional task was engaged, since the
    sequence exists to describe optional-task behaviour.
    """
    tokens = []
    final_answer_idx = 0
    opt_used = False
    final_answer_pending = True

    for step, action, outcome in zip(steps, actions, outcomes):
        previous_step = tokens[-1].split("-")[0] if tokens else ""
        new_step = not tokens or step != previous_step

        if new_step and step in OPT_ALL_STEPS:
            tokens.append(step)
            if step in OPT_SUBSTEPS:
                opt_used = True
            continue

        if action == "Attempt" and outcome != "OK":
            token = step + "-2"
        elif "Hint" in str(action):
            token = step + "-1"
        else:
            token = step + "-0"

        if new_step:
            if step == "FinalAnswer" and opt_used and final_answer_pending:
                final_answer_idx = len(tokens)
                final_answer_pending = False
            tokens.append(token)
        elif step not in OPT_ALL_STEPS and tokens[-1] < token:
            tokens[-1] = token          # keep the worst code seen on this step

    if not (opt_used and tokens):
        return ""
    head = tokens[:final_answer_idx + 1]
    head[-1] = "FinalAnswer"
    return "\t".join(head)


def time_info(steps, times):
    """Inter-step gaps, split by optional/non-optional, up to the post-optional answer.

    Returns (seq_time_vec, opt_step_time, non_opt_step_time, actual_fa_opt_time, total).
    `actual_fa_opt_time` is how long the student took from their last optional-task step to
    submitting the final answer.
    """
    seq, opt_times, non_opt_times = [], [], []
    opt_used = False
    final_answer_idx = last_opt_idx = -1

    for i, (step, time) in enumerate(zip(steps, times)):
        delta = (time - times[i - 1]) if i > 0 else 0
        if step in OPT_SUBSTEPS:
            opt_used = True
            last_opt_idx = i
            opt_times.append(delta)
        else:
            non_opt_times.append(delta)
        seq.append(delta)
        if opt_used and step == "FinalAnswer":
            final_answer_idx = i
            break

    fa_opt_time = 0
    if final_answer_idx != -1 and last_opt_idx != -1:
        fa_opt_time = times[final_answer_idx] - times[last_opt_idx]
    return seq, opt_times, non_opt_times, fa_opt_time, sum(seq)


def optional_task_flags(group: pd.DataFrame):
    """(every optional step right first time, final answer correct after optional work)."""
    opt_first = (group[group["Step Name"].isin(OPT_SUBSTEPS)]
                 .drop_duplicates("Step Name", keep="first"))
    all_correct = bool(not opt_first.empty and ((opt_first["Outcome"] == "OK")
                                                & (opt_first["Action"] == "Attempt")
                                                & (opt_first["Attempt At Step"] == 1)).all())
    correctness = 0
    if set(group["Step Name"]) & OPT_SUBSTEPS:
        final_answer = group[group["Step Name"] == "FinalAnswer"]
        if not final_answer.empty and final_answer.iloc[0]["Outcome"] == "OK":
            correctness = 1
    return all_correct, correctness


def export_problem_info(path, df: pd.DataFrame, prob_type_map: dict, scenario_map: dict,
           prob_er_me: dict, path_info: dict, etalon_map: dict,
           verbose: bool = True) -> int:
    """Write problem_info.csv and return the number of rows written."""
    genuine = df[(df["Action"] == "Attempt") & df["Outcome"].isin(GENUINE_OUTCOMES)]
    order = (genuine.groupby(["Anon Student Id", "Problem Name"])["Time"]
             .min().reset_index()
             .sort_values(["Anon Student Id", "Time"]))
    total_problems = order.groupby("Anon Student Id")["Problem Name"].nunique().to_dict()

    student_meta = (df.groupby("Anon Student Id")
                    .agg(school=("CF (Anon School Id)", "first"),
                         progress=("CF (Workspace Progress Status)", "first"))
                    .to_dict("index"))

    # p-Known snapshots are indexed by the workspace's own mastery KC list.
    kcs = sorted(df["KC Model(MATHia)"].dropna().unique())
    kc_index = {kc: i for i, kc in enumerate(kcs)}

    non_autofilled = df[df["CF (Is Autofilled)"] == False]  # noqa: E712
    columns = [c for c in TRANSACTION_COLS if c in non_autofilled.columns]
    groups = dict(tuple(non_autofilled[columns].groupby(["Anon Student Id", "Problem Name"])))

    n_rows = 0
    with open(path, "w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(HEADER)

        for student, student_problems in order.groupby("Anon Student Id"):
            meta = student_meta.get(student, {})
            for count, (_, prow) in enumerate(
                    student_problems.sort_values("Time").iterrows(), 1):
                problem = prow["Problem Name"]
                group = groups.get((student, problem))
                if group is None or group.empty:
                    continue
                group = group.sort_values("Time")
                row = _build_row(student, problem, count, group, meta,
                                 total_problems.get(student, 0), prob_type_map,
                                 scenario_map, prob_er_me, path_info, etalon_map,
                                 kcs, kc_index)
                writer.writerow(row)
                n_rows += 1

    if verbose:
        print(f"problem_info.csv saved -> {path}")
        print(f"  {n_rows} rows (student-problem pairs)")
    return n_rows


def _build_row(student, problem, count, group, meta, n_total, prob_type_map, scenario_map,
               prob_er_me, path_info, etalon_map, kcs, kc_index):
    er_me = prob_er_me.get(problem, -1)
    has_er, has_me = path_info.get((student, problem), (False, False))

    clean = group.dropna(subset=["Step Name"])
    step_names = list(clean["Step Name"])
    times = list(clean["Time"])
    step_set = set(step_names)

    # One record per raw transaction, in the order they happened.
    records = (clean["Step Name"].astype(str) + "-"
               + clean["Action"].astype(str) + "-"
               + clean["Attempt At Step"].astype(str) + "-"
               + clean["Help Level"].astype(str) + "-"
               + clean["Outcome"].astype(str) + "-"
               + clean["Time"].astype(str)).tolist()

    all_opt_correct, fa_correctness = optional_task_flags(clean)
    tokenized = tokenize_steps(step_names, list(clean["Action"]), list(clean["Outcome"]))
    seq_time, opt_time, non_opt_time, fa_opt_time, total_time = time_info(step_names, times)

    # Snapshot of MATHia's own mastery estimates before and after each attempt.
    prev_known = [0] * len(kcs)
    new_known = [0] * len(kcs)
    fa_skill_index = -1
    attempts = clean[(clean["Action"] == "Attempt") & clean["KC Model(MATHia)"].notna()]
    for kc, previous, updated, step in zip(attempts["KC Model(MATHia)"],
                                          attempts["CF (Skill Previous p-Known)"],
                                          attempts["CF (Skill New p-Known)"],
                                          attempts["Step Name"]):
        index = kc_index.get(kc)
        if index is not None:
            prev_known[index] = previous
            new_known[index] = updated
            if step == "FinalAnswer":
                fa_skill_index = index

    return [
        meta.get("school", ""), meta.get("progress", ""), student, count, n_total,
        problem, prob_type_map.get(problem, ""), scenario_map.get(problem, ""),
        "ME" if er_me == 1 else ("ER" if er_me == 0 else "UNK"),
        etalon_map.get(problem, ""),
        compute_label_opt(step_set), all_opt_correct, fa_correctness,
        len(step_set), len(records), "\t".join(records), tokenized,
        "\t".join(str(v) for v in prev_known),
        "\t".join(str(v) for v in new_known),
        fa_skill_index,
        "\t".join(str(v) for v in seq_time),
        "\t".join(str(v) for v in opt_time),
        "\t".join(str(v) for v in non_opt_time),
        fa_opt_time, total_time,
        recognize_response(er_me, has_er, has_me),
    ]

## Reconstruction

Rebuild raw MATHia transactions from `problem_info.csv`, for chosen students or schools.

`problem_info.csv` stores each attempt's full transaction sequence in its tokenized columns, so the rows can be regenerated without the original export.

Every rebuilt attempt is checked back against the columns `problem_info.csv` derived from the original rows -- step order, timings, engagement labels, token sequence -- and the check fails loudly if any disagree. Columns MATHia exports but `problem_info.csv` does not retain (class id, p-Known snapshots) come back empty.

In [ ]:
#!/usr/bin/env python3




csv.field_size_limit(10 ** 9)

# problem_info.csv columns that must be numeric for the comparisons below.
NUMERIC_COLS = ["count", "total_problems", "label_opt", "n_tokenized_steps",
                "n_original_steps", "fa_skill_index", "fa_correctness_after_opt",
                "actual_fa_opt_time", "tot_prob_time", "recognize_response"]

# Values every reconstructed row carries, since the pipeline only keeps rows with these.
FILL_CONSTANTS = {
    "CF (Is StepByStep)": False,
    "CF (Encounter)": 0,
    "CF (Is Review Mode)": -1,
    "CF (Is Autofilled)": False,
}


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("workspace", choices=sorted(WORKSPACES) + sorted(ALIASES))
    parser.add_argument("--ids", nargs="+", required=True,
                        help="student ids (stu_*) and/or school ids (sch_*) to rebuild")
    parser.add_argument("--output-dir", type=str, default=None,
                        help=f"pipeline output root (default: {OUTPUT_DIR})")
    parser.add_argument("--no-check", action="store_true",
                        help="skip the fidelity check against problem_info.csv")
    return parser.parse_args(argv)


def load_rows(path, ids):
    """Read only the problem_info.csv rows belonging to the requested students/schools."""
    wanted = set(ids)
    with open(path) as handle:
        header_line = handle.readline()
        # Substring prescan first: the transaction column makes these rows very wide.
        kept = [line for line in handle if any(i in line for i in wanted)]

    header = next(csv.reader([header_line]))
    rows = list(csv.reader(io.StringIO("".join(kept))))
    malformed = [r for r in rows if len(r) != len(header)]
    if malformed:
        raise ValueError(f"{len(malformed)} malformed rows in {path}")

    df = pd.DataFrame(rows, columns=header)
    df = df[df["student"].isin(wanted) | df["school"].isin(wanted)].copy()
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.sort_values(["school", "student", "count"]).reset_index(drop=True)


def parse_token(token):
    """`<step>-<action>-<attempt>-<help level>-<outcome>-<time>` -> its six fields."""
    step, action, attempt, help_level, outcome, time = token.rsplit("-", 5)
    return (step, action,
            int(attempt) if attempt.isdigit() else np.nan,
            int(help_level) if help_level.isdigit() else np.nan,
            outcome, int(time))


def explode_transactions(info, columns=MATHIA_COLUMNS):
    """One row per transaction token, in MATHia's column layout."""
    records = []
    for attempt in info.itertuples(index=False):
        for token in attempt.original_steps.split("\t"):
            step, action, attempt_at_step, help_level, outcome, time = parse_token(token)
            row = dict.fromkeys(columns, "")
            row.update({k: v for k, v in {
                "Anon Student Id": attempt.student,
                "Time": time,
                "Problem Name": attempt.problem,
                "Step Name": step,
                "Attempt At Step": attempt_at_step,
                "Outcome": outcome,
                "Action": action,
                "Help Level": help_level,
                # Only NumeratorFactor's etalon is retained, and only per problem.
                "CF (Etalon)": attempt.etalon_value if step == "NumeratorFactor" else "",
                "CF (Anon School Id)": attempt.school,
                "CF (Workspace Progress Status)": attempt.progress,
                **FILL_CONSTANTS,
            }.items() if k in row})
            records.append(row)

    raw = pd.DataFrame(records, columns=columns)
    return (raw.sort_values(["Anon Student Id", "Time"], kind="stable")
            .reset_index(drop=True))


def check_fidelity(info, raw):
    """Recompute every derived problem_info.csv column from the rebuilt rows."""
    groups = dict(tuple(raw.groupby(["Anon Student Id", "Problem Name"], sort=False)))

    checks = []
    for attempt in info.itertuples(index=False):
        group = groups[(attempt.student, attempt.problem)]
        steps, times = list(group["Step Name"]), list(group["Time"])
        step_set = set(steps)
        seq, _, _, fa_opt_time, total_time = time_info(steps, times)
        all_opt_correct, fa_correctness = optional_task_flags(group)

        has_er = bool(step_set & ER_PATH_STEPS)
        has_me = bool(step_set & ME_PATH_STEPS)
        er_me = {"ER": 0, "ME": 1}.get(attempt.er_me, -1)

        stored_seq = ([int(v) for v in attempt.seq_time_vec.split("\t")]
                      if attempt.seq_time_vec else [])
        stored_steps = [t.rsplit("-", 5)[0] for t in attempt.original_steps.split("\t")]

        checks.append({
            "school": attempt.school, "student": attempt.student,
            "count": attempt.count, "problem": attempt.problem,
            "step_order": steps == stored_steps,
            "time_monotonic": all(b >= a for a, b in zip(times, times[1:])),
            "n_rows": len(group) == attempt.n_original_steps,
            "n_unique_steps": len(step_set) == attempt.n_tokenized_steps,
            "seq_time_vec": seq == stored_seq,
            "tot_prob_time": total_time == attempt.tot_prob_time,
            "actual_fa_opt_time": fa_opt_time == attempt.actual_fa_opt_time,
            "label_opt": compute_label_opt(step_set) == attempt.label_opt,
            "all_opt_correct": str(all_opt_correct) == str(attempt.all_opt_correct),
            "fa_correctness": fa_correctness == attempt.fa_correctness_after_opt,
            "tokenized_steps": tokenize_steps(steps, list(group["Action"]),
                                              list(group["Outcome"]))
                               == (attempt.tokenized_steps or ""),
            "recognize_response": (recognize_response(er_me, has_er, has_me)
                                   == attempt.recognize_response),
        })

    return pd.DataFrame(checks)


def report_fidelity(checks, raw):
    """Print pass/fail per derived column; return True when everything matched."""
    flags = [c for c in checks.columns
             if c not in ("school", "student", "count", "problem")]
    print(f"\nFidelity over {len(checks)} problems:")
    for flag in flags:
        n_ok = int(checks[flag].sum())
        print(f"  {'PASS' if n_ok == len(checks) else 'FAIL'}  {flag:<20} "
              f"{n_ok}/{len(checks)}")

    monotonic = raw.groupby("Anon Student Id")["Time"].apply(lambda s: s.is_monotonic_increasing)
    print(f"\nstudents with non-decreasing Time across file: "
          f"{int(monotonic.sum())}/{len(monotonic)}")

    failed = checks[~checks[flags].all(axis=1)]
    if not failed.empty:
        print(f"\n{len(failed)} problem(s) failed:")
        print(failed.to_string(index=False))
    return failed.empty


def write_csv(raw, out_dir, ids):
    """Write the rebuilt rows, tagged by selection and timestamp so runs never overwrite."""
    out_dir.mkdir(parents=True, exist_ok=True)
    n_students = raw["Anon Student Id"].nunique()
    tag = ids[0] if len(ids) == 1 else f"{len(ids)}ids{n_students}stu"
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    path = out_dir / f"{tag}_raw_reconstructed_{stamp}.csv"
    duplicate = 1
    while path.exists():
        path = out_dir / f"{tag}_raw_reconstructed_{stamp}_{duplicate}.csv"
        duplicate += 1

    raw.to_csv(path, index=False)
    print(f"\n{path}  ({path.stat().st_size / 1024:.1f} KB, "
          f"{len(raw)} rows x {len(raw.columns)} cols, {n_students} students)")
    return path


def main(argv=None):
    args = parse_args(argv)
    ws = get_workspace(args.workspace)
    root = OUTPUT_DIR if args.output_dir is None else Path(args.output_dir)
    info_path = root / ws.name / "problem_info.csv"
    if not info_path.exists():
        raise FileNotFoundError(f"{info_path} not found -- run "
                                f"notebook 1 first.")

    print(f"workspace: {ws.name}")
    print(f"source:    {info_path}")
    print(f"select:    {args.ids}")

    info = load_rows(info_path, args.ids)
    if info.empty:
        raise ValueError(f"no rows for {args.ids} in {info_path}")

    unmatched = [i for i in args.ids
                 if i not in set(info["student"]) | set(info["school"])]
    if unmatched:
        print(f"no match in student/school columns: {unmatched}")

    print(f"\nproblem_info rows: {len(info)}")
    print(f"schools:  {info['school'].nunique()}")
    print(f"students: {info['student'].nunique()}")
    print(f"prob_type: {info['prob_type'].value_counts().to_dict()}")
    print(f"transactions to rebuild: {int(info['n_original_steps'].sum())}")

    raw = explode_transactions(info, ws.columns)
    filled = [c for c in ws.columns if not (raw[c] == "").all()]
    print(f"\nreconstructed transactions: {len(raw)}")
    print(f"  actions:  {raw['Action'].value_counts().to_dict()}")
    print(f"  outcomes: {raw['Outcome'].value_counts().to_dict()}")
    print(f"  filled columns ({len(filled)}): {filled}")
    print(f"  empty columns:  {[c for c in ws.columns if c not in filled]}")

    ok = True
    if not args.no_check:
        ok = report_fidelity(check_fidelity(info, raw), raw)

    write_csv(raw, root / ws.name / "reconstructed", args.ids)
    return 0 if ok else 1


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (FileNotFoundError, ValueError) as error:
        sys.exit(f"error: {error}")

### Run

Pick students or whole schools by id.

In [ ]:
WORKSPACE = "change3"
IDS = []          # e.g. ["stu_AAAPM61509"]; empty selects the first few students present

ws = get_workspace(WORKSPACE)
info_path = workspace_dir(ws.name) / "problem_info.csv"
ids = IDS or sorted(pd.read_csv(info_path, usecols=["student"])["student"].unique())[:2]
print(f"reconstructing {ids}")

main([WORKSPACE, "--ids", *ids, "--output-dir", str(OUTPUT_DIR)])